# Reusable template — fiscal top-income panel

**Short name:** `TopInc_Py`  
Copy this notebook when you receive another WTID-style file (country × year × named fractile columns). Change the knobs, not the helpers.


In [ ]:
import csv
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- knobs ----
DATA = Path("data/income_dist.csv")
COUNTRY = "United States"
SHARE_COLS = [
    "Top 10% income share",
    "Top 5% income share",
    "Top 1% income share",
    "Top 0.5% income share",
    "Top 0.1% income share",
]
COMPARE = ["United States", "France", "Spain", "Sweden"]
START, END = 1980, 2007
NOISE_SD = 0.30
# ---------------

def _to_float(x):
    try:
        return None if x is None or str(x).strip() == "" else float(x)
    except ValueError:
        return None

def load(path):
    return pd.read_csv(path)

def country_frame(df, name):
    return df[df["Country"] == name].sort_values("Year")

def plot_shares(frame, cols, title):
    ax = frame.set_index("Year")[cols].plot(lw=2, figsize=(10, 4.4))
    ax.set_ylabel("% of total income")
    ax.set_title(title)
    plt.tight_layout(); plt.show()

def endpoint_change(frame, col, start, end):
    v0 = frame.loc[frame.Year == start, col]
    v1 = frame.loc[frame.Year == end, col]
    if v0.empty or v1.empty:
        return None
    return float(v1.iloc[0] - v0.iloc[0])

df = load(DATA)
us = country_frame(df, COUNTRY)
plot_shares(us, [c for c in SHARE_COLS if c in us.columns], f"{COUNTRY} top shares")
print("change", START, END, endpoint_change(us, "Top 1% income share", START, END))


## Cross-section helper

Reuse `COMPARE` / `START` / `END`. If a country has no exact year, walk ±2 years.


In [ ]:
def nearest_value(frame, year, col, window=2):
    sl = frame.loc[frame.Year.between(year-window, year+window), ["Year", col]].dropna()
    if sl.empty:
        return None
    row = sl.iloc[(sl.Year - year).abs().argmin()]
    return int(row.Year), float(row[col])

rows = []
for name in COMPARE:
    sub = country_frame(df, name)
    a = nearest_value(sub, START, "Top 1% income share")
    b = nearest_value(sub, END, "Top 1% income share")
    if a and b:
        rows.append({"country": name, "y0": a[0], "v0": a[1], "y1": b[0], "v1": b[1], "dpp": b[1]-a[1]})
print(pd.DataFrame(rows).to_string(index=False))


## Simulation knobs

Same contract as the solution notebook. Swap `COUNTRY` to stress-test a smaller change.


In [ ]:
rng = np.random.default_rng(0)
col = "Top 1% income share"
sub = country_frame(df, COUNTRY)[["Year", col]].dropna()
v0 = float(sub.loc[sub.Year == START, col].iloc[0])
v1 = float(sub.loc[sub.Year == END, col].iloc[0])
draws = (v1 + rng.normal(0, NOISE_SD, 2000)) - (v0 + rng.normal(0, NOISE_SD, 2000))
print(f"{COUNTRY} {START}->{END}: point {v1-v0:.2f} pp; sim 5-95 {np.quantile(draws,[0.05,0.95])}")
